In [1]:
from datetime import datetime, timedelta
import os
from typing import List
import xarray as xr
import json
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import sys
sys.path.append(".")  # Add current directory to Python path
from utils import filename_to_year, datetime_range, open_hdf5

In [3]:
# Assign experiment number - this was for my own organization
experiment_number = 1

# steps are 6-hourly (e.g. 12 steps = 3 days)
n_steps = 20

# Select timestep to save - string format yyyy-mm-ddTHH:MM:SS
init_timestep = None
valid_timestep = '2022-10-14T00:00:00'#'2019-03-22T12:00:00'

if valid_timestep is None:
    valid_timestep = (datetime.fromisoformat(init_timestep) + timedelta(hours = n_steps*6)).isoformat() 
elif init_timestep is None:
    init_timestep = (datetime.fromisoformat(valid_timestep) - timedelta(hours = n_steps*6)).isoformat()

# Filepath for ERA5 json data
SFNO_dir = "/projectnb/eb-general/shared_data/data/processed/FourCastNet_sfno/ERA5_SFNO/testing_data"
data_fp = os.path.join(SFNO_dir, 'data.json') #"/barnes-engr-scratch2/C837824079/ERA5_SFNO/data.json"

# Filepath for init_file
init_fp = os.path.join('/INSERT_INFERENCE_RUNS_DIRECTORY/Experiment1/initialization_files', f"Initialize_{datetime.fromisoformat(init_timestep).strftime("%Y_%m_%dT%H")}_nsteps{n_steps}.nc")


print(f"Selecting timestep {init_timestep} to {valid_timestep}")

Selecting timestep 2022-10-09T00:00:00 to 2022-10-14T00:00:00


In [4]:
# Open and load the JSON file
with open(data_fp, 'r') as f:
    labels = json.load(f)

# open initial conditions from stored ERA data
year_of_timestep = datetime.fromisoformat(init_timestep).year
data_create = open_hdf5(path = os.path.join(SFNO_dir, str(year_of_timestep)+'.h5'), metadata = labels)
data_create = data_create.sel(time = [init_timestep, valid_timestep]) # this just selects the first and last time in the time range
data_create = data_create.rename({"channel": "variable"})
data_create


<xarray.DataArray 'fields' (time: 2, variable: 74, lat: 721, lon: 1440)> Size: 615MB
[153659520 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 16B 2022-10-09 2022-10-14
  * variable   (variable) <U5 1kB 'u10m' 'v10m' 'u100m' ... 'q925' 'q1000'
  * lat        (lat) float64 6kB 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * lon        (lon) float64 12kB 0.0 0.25 0.5 0.75 ... 359.0 359.2 359.5 359.8
    grid_type  <U11 44B 'equiangular'
Attributes:
    description:  ERA5 data at 6 hourly frequency with snapshots at 0000, 060...
    path:         /projectnb/eb-general/shared_data/data/processed/FourCastNe...

In [ ]:
data_create.to_netcdf(init_fp)